# CMB-HD simulation example patches
This notebook is intended to quickly generate CMB patches that are $2\degree$ by $2\degree$. While the full-sized patches are $12\degree$ by $12\degree$, we use smaller patches in this tutorial so that simulated maps and power spectra can be calculated *in* the notebook (there are some exceptions to this, which are pointed out). We note that the power spectra resulting from these smaller patches will vary from the full-sized patches because of patch-to-patch variance.  

Full-size patches and patch power spectra are more computationally intensive and can be generated using the `.py` files referenced in this notebook as you work through it.

In this notebook, we demonstrate how to generate all CMB foregrounds, as some are treated slightly differently (diffuse vs discrete foregrounds, scaling for the tSZ and CIB, and the lack of a pixel window treatment for kappa). But we only use a single frequency (90GHz) for brevity. To use this notebook for different frequencies, just change that value (or refer to the full-sized python scripts).

To generate these patches, we need access to the original 2010 simulations from [https://lambda.gsfc.nasa.gov/simulation/full_sky_sims_ov.html](https://lambda.gsfc.nasa.gov/simulation/full_sky_sims_ov.html). To run this notebook, we have provided some sky patches and catalogs already in the `hdsims_output_from_example_notebook/S10_patches` and `hdsims_output_from_example_notebook/binning_files` folders, as computing them takes a long time. If you want to generate those yourself (for example, picking a different center RA, Dec), uncomment out the cells which are commented out and use `source download_90GHz_S10_data.sh` in the `S10_data/` folder to download the necessary files from the original 2010 simulations. If you want to truly start from scratch to generate full-sized patches, you should use `source download_all_S10_data.sh` in the `S10_data/` folder, then run `deconvolve_all_S10_fullsky.py` and `12x12_reduce_S10_catalogs.py` in order to deconvolve the full-sky maps and isolate the point sources in your desired patch. But be warned that these can take a long time (on the order of hours), so we don't recommend it.

### Output Directories

There are three pieces to import: Foregrounds (which handles everything regarding generating the foregrounds), Spectra (which handles generating binning files and taking the power spectra for any patch of sky), and CMB (which handles generating the T,Q,U unlensed patches of sky and using the lensing convergence map for lensing).

In [2]:
from hdsims import Foregrounds, Spectra, CMB

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import healpy as hp
import os
import gc
from matplotlib.colors import LinearSegmentedColormap
from pixell import enmap, enplot, colorize, utils, wcsutils, curvedsky as cs
from pspy import so_map, so_mcm, so_spectra, pspy_utils
import useful_plots as upl
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams.update(mpl.rcParamsDefault)
plt.rcParams['figure.dpi'] = 250
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.xmargin'] = 0.025
plt.rcParams['axes.ymargin'] = 0.025
plt.rcParams['grid.alpha'] = 0.2
plt.rcParams['figure.figsize'] = (5, 3)
if 'planck' not in mpl.colormaps:
    colorize.mpl_setdefault('planck')

planck_cmap = plt.get_cmap('planck')
pos_cmap = upl.truncate_colormap(planck_cmap, minval=0.5, maxval=1.0, n=100)
neg_cmap = upl.truncate_colormap(planck_cmap, minval=0.0, maxval=0.5, n=100)

overall_path_example = %pwd
overall_path_example += '/hdsims_output_from_example_notebook/'
# this is the path where results from/for this notebook are located

S10_resolution = hp.nside2resol(8192, arcmin=True) # Original S10 0.43' pixel size
S10_resolution_kappa = hp.nside2resol(4096, arcmin=True) # Original S10 0.86' pixel size

#### Binning Files

These are used for everything related to taking power spectra:

In [ ]:
spectra_S10 = Spectra(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    res = S10_resolution,
    apod_width = 0.2,
    l_max = 12574)
spectra_S10_kappa = Spectra(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    res = S10_resolution_kappa,
    apod_width = 0.2,
    l_max = 6287)
spectra_HD = Spectra(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    res = 0.04,
    apod_width = 0.2,
    l_max = 24000)

In [ ]:
# For the specified patch sizes above, these generate the appropriate binning files. For some, this involves
# Cl or Dl, and for the CMB patch, this involves spin0and2 patches rather than just spin0. We've included
# the relative outputs in `hdsims_output_from_example_notebook/binning_files`
'''
spectra_S10.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/")
spectra_S10_kappa.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/", type_Cl = True)
spectra_S10_kappa.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/")
spectra_HD.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/")
spectra_HD.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/", type_Cl = True)
spectra_HD.make_binning_files(binning_output_path = f"{overall_path_example}binning_files/", spin0and2 = True)
'''

For the full-sized patch, use `12x12_Binning_Files.py`.

And these are used for everything related to making foregrounds (radio, CIB, kSZ, tSZ, and kappa):

In [ ]:
foregrounds_HD = Foregrounds(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    new_res = 0.04,
    apod_width = 0.2,
    l_max = 24000)
foregrounds_HD_kappa = Foregrounds(
    ra = 6,
    dec = 6,
    final_width = 2.4,
    new_res = 0.04,
    apod_width = 0.2,
    l_max = 24000)

Note that we use a different foreground object for kappa as compared to the rest of the foregrounds. This is because we want a final lensed CMB that is $2\degree$ by $2\degree$, which necessitates that we do some work on the lensing convergence map that requires it to be apodized. For that reason, we begin with a kappa patch that is $2.4\degree$ by $2.4\degree$.

### HD Foregrounds

Note that we typically set `l_max = 24000` even though we only care up to `l_max = 20000`, as there are inaccuracies at the very end of the computation limit.

If you want to are going to generate the $2^\degree$ by $2^\degree$ patches fully from scratch (and uncommment out the relevant parts), you'll first need to run `deconvolve_90GHz_S10_fullsky.py`, which takes the 90GHz S10 fullsky maps and deconvolves them with the pixel window (except for the lensing convergence). You'll also need to run `example_reduce_S10_catalogs.py`, which takes the Lambda radio and CIB catalogs and isolates the ones in the chosen small patch.

Each foreground section here is self-contained, so you can run them in any order. We also clear each foreground's variables after its section to clear up memory.

To do the same tasks for the full-size patches, use `12x12_HD_Diffuse_Foregrounds.py`, `12x12_HD_Discrete_Foregrounds.py`, and `12x12_HD_CMB.py`.

#### tSZ

In [ ]:
# Upsampling S10 sim
HD_tSZ_patch = foregrounds_HD.generate_diffuse_foreground(
                component = 'tSZ',
                frequency = 90,
                S10_largeApodized_path = f"{overall_path_example}S10_patches/tSZ_90GHz_2.4x2.4deg_ra=6_dec=6_official")
HD_tSZ_patch.write_map(f"{overall_path_example}tSZ_90GHz_2x2deg_ra=6_dec=6")

In [ ]:
# Taking power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/")
HD_tSZ_dls, HD_tSZ_ells = spectra_HD.get_foreground_power(HD_tSZ_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}tSZ_90GHz_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_tSZ_ells, "dl": HD_tSZ_dls})

In [ ]:
del HD_tSZ_patch
del mbb_inv, binning_file, Bbl
del HD_tSZ_dls, HD_tSZ_ells
gc.collect()

#### kSZ

In [ ]:
# Upsampling S10 sim
S10_kSZ_patch = foregrounds_HD.generate_diffuse_foreground(
                component = 'kSZ',
                frequency = 90,
                S10_largeApodized_path = f"{overall_path_example}S10_patches/kSZ_90GHz_2.4x2.4deg_ra=6_dec=6_official")

# We'll need a theory spectra to use as a template, which we include in `S10_data/`
ksz_template = pd.read_csv("S10_data/cmbhd_mockdata_ksz_cls_v1.1.txt", delimiter=' ')
template_ells = ksz_template['ell']
template_cls = ksz_template['C_ell^kSZ']

# Extending to small-scales
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/", type_Cl = True)
HD_kSZ_patch, kSZ_theory_power = foregrounds_HD_kappa.extend_to_small_scales(S10_patch = S10_kSZ_patch, patch_type='kSZ', mbb_inv=mbb_inv, binning_file=binning_file)
HD_kSZ_patch.write_map(f"{overall_path_example}kSZ_{frequency}GHz_2x2deg_ra=6_dec=6")
np.save(f"{overall_path_example}kSZ_{frequency}GHz_smallscale_theory_spectra.npy", kSZ_theory_power)

In [ ]:
# Taking power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/")
HD_kSZ_dls, HD_kSZ_ells = spectra_HD.get_foreground_power(HD_kSZ_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}kSZ_{frequency}GHz_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_kSZ_ells, "dl": HD_kSZ_dls})

In [ ]:
del S10_kSZ_patch
del ksz_template, template_ells, template_cls
del mbb_inv, binning_file, Bbl
del HD_kSZ_patch, kSZ_theory_power
del HD_kSZ_dls, HD_kSZ_ells
gc.collect()

#### Lensing Convergence (Kappa)

In [ ]:
# Upsampling S10 sim
S10_kappa_patch = foregrounds_HD_kappa.generate_diffuse_foreground(
                  component = 'kappa',
                  frequency = None,
                  S10_largeApodized_path = f"{overall_path_example}S10_patches/kappa_2.6x2.6deg_ra=6_dec=6_official")

# We'll need a theory spectra to use as a template
main_data_text, main_data_dat = foregrounds_HD_kappa.get_kappa_theory()
np.savetxt(f'{overall_path_example}camb_full_output.txt', main_data_text,
    header='ell    tt_lensed   ee_lensed   bb_lensed   te_lensed   '
           'tt_unlensed ee_unlensed bb_unlensed te_unlensed kk',
    fmt='%d %.6e %.6e %.6e %.6e %.6e %.6e %.6e %.6e %.6e')
np.savetxt(f'{overall_path_example}/camb_full_output.dat', main_data_dat, fmt='%.6f', delimiter='\t')
hddata = np.loadtxt(f"{overall_path_example}camb_full_output.txt", comments="#")
template_ells =  hddata[:,0]
template_cls = hddata[:,9]
template_cls *= 2*np.pi/(template_ells * (template_ells + 1))**2

# Extending to small-scales
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/", type_Cl = True)
HD_kappa_patch, kappa_theory_power = foregrounds_HD_kappa.extend_to_small_scales(S10_patch = S10_kappa_patch, patch_type='kappa', mbb_inv=mbb_inv, binning_file=binning_file)
HD_kappa_patch.write_map(f"{overall_path_example}kappa_2.4x2.4deg_ra=6_dec=6")
np.save(f"{overall_path_example}kappa_smallscale_theory_spectra.npy", kappa_theory_power)

In [ ]:
# Taking power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/", type_Cl = True)
HD_kappa_cls, HD_kappa_ells = spectra_HD.get_foreground_power(HD_kappa_patch, mbb_inv, binning_file, deconvolve_pw = False, type_Cl = True)
np.save(f"{overall_path_example}kappa_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_kappa_ells, "cl": HD_kappa_cls})

In [ ]:
del S10_kappa_patch
del main_data_text, main_data_dat, hddata, template_ells, template_cls
del mbb_inv, binning_file, Bbl
del HD_kappa_patch, kappa_theory_power
del HD_kappa_cls, HD_kappa_ells
gc.collect()

#### Radio Sources

In [ ]:
# Making high-resolution sim
HD_radio_patch = foregrounds_HD.generate_discrete_foreground(
                            frequency = 90,
                            catalog = pd.read_csv(f"{overall_path_example}S10_patches/radio_2.4x2.4deg_source_catalog_ra=6_dec=6_official.csv"))
HD_radio_patch.write_map(f"{overall_path_example}radio_{frequency}GHz_2x2deg_ra=6_dec=6")

In [ ]:
# Taking power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/")
HD_radio_dls, HD_radio_ells = spectra_HD.get_foreground_power(HD_radio_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}radio_{frequency}GHz_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_radio_ells, "dl": HD_radio_dls})

In [ ]:
del HD_radio_patch
del mbb_inv, binning_file, Bbl
del HD_radio_dls, HD_radio_ells
gc.collect()

#### CIB

In [ ]:
# Making catalog for CIB model
CIB_catalog = foregrounds_HD.make_CIB_model_catalog(CIB_model=1, CIB_catalog_original = pd.read_csv(f"{overall_path_example}S10_patches/CIB_14x14deg_source_catalog_ra=6_dec=6.csv"))

# Making high-resolution sim
HD_CIB_patch = foregrounds_HD.generate_discrete_foreground(
                                frequency = 90,
                                catalog = CIB_catalog)
HD_CIB_patch.write_map(f"{overall_path_example}CIB_{frequency}GHz_2x2deg_ra=6_dec=6")

In [ ]:
# Taking power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/")
HD_CIB_dls, HD_CIB_ells = spectra_HD.get_foreground_power(HD_CIB_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}CIB_{frequency}GHz_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_CIB_ells, "dl": HD_CIB_dls})

In [ ]:
del CIB_catalog
del HD_CIB_patch
del mbb_inv, binning_file, Bbl
del HD_CIB_dls, HD_CIB_ells
gc.collect()

#### CMB (Lensed)

In [ ]:
CMB_patch = CMB(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    res = 0.04,
    apod_width = 0.2,
    l_max = 24000)

# Make high-resolution sim for unlensed CMB (at width of final_width + 2*apod_width)
cmb_unlensed = CMB_patch.make_unlensed_patch(theory_path = f"{overall_path_example}camb_full_output.dat", 
                                  cmb_seed = 58)

# Lens with the 2.4x2.4 kappa patch from earlier
HD_kappa_patch = so_map.read_map(f"{overall_path_example}kappa_2.4x2.4deg_ra=6_dec=6")
cmb_lensed = CMB_patch.do_lensing(cmb_unlensed, HD_kappa_patch)
cmb_lensed.write_map(f"{overall_path_example}CMB_Lensed_2x2deg_ra=6_dec=6")

In [ ]:
# Take power spectra
mbb_inv, binning_file, Bbl = spectra_HD.get_binning_files(path = f"{overall_path_example}binning_files/", spin0and2 = True)
HD_lensedCMB_dls, HD_lensedCMB_ells = spectra_HD.get_CMB_power(cmb_lensed, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}CMB_Lensed_2x2deg_ra=6_dec=6_spectra.npy", {"l": HD_lensedCMB_ells, "dl": HD_lensedCMB_dls})

In [ ]:
del cmb_unlensed
del HD_kappa_patch, cmb_lensed
del mbb_inv, binning_file, Bbl
del HD_lensedCMB_dls, HD_lensedCMB_ells
gc.collect()

### Plots

In all the above code, we've been working with .04' maps. When it comes to plotting our results, we often want to compare against the original S10 simulations' power, so this section generates those .43' and .86' maps, and takes their power spectra. We've included these in `hdsims_output_from_example_notebook/S10_patches`, so we've commented out the sections to generate them.

For the full-sized patches, use `12x12_S10_Foregrounds.py`.

In [ ]:
'''
foregrounds_S10 = Foregrounds(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    new_res = S10_resolution,
    apod_width = 0.2,
    l_max = 12574)

foregrounds_S10_kappa = Foregrounds(
    ra = 6,
    dec = 6,
    final_width = 2.0,
    new_res = S10_resolution_kappa,
    apod_width = 0.2,
    l_max = 6827)
'''

In [ ]:
'''
mbb_inv, binning_file, Bbl = spectra_S10.get_binning_files(path = f"{overall_path_example}binning_files/")

# kSZ
S10_kSZ_patch = foregrounds_S10.generate_diffuse_foreground(
                component = 'kSZ',
                frequency = 90,
                fullsky_deconvolved_path = f"S10_data/deconvolved_fullsky/{frequency}GHz_kSZ_fullsky_deconvolved")
S10_kSZ_dls, S10_kSZ_ells = spectra_S10.get_foreground_power(S10_kSZ_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}S10_patches/kSZ_90GHz_2x2deg_ra=6_dec=6_S10spectra.npy", {"l": S10_kSZ_ells, "dl": S10_kSZ_dls})

# tSZ
S10_tSZ_patch = foregrounds_S10.generate_diffuse_foreground(
                component = 'tSZ',
                frequency = 90,
                fullsky_deconvolved_path = f"S10_data/deconvolved_fullsky/{frequency}GHz_tSZ_fullsky_deconvolved")
S10_tSZ_dls, S10_tSZ_ells = spectra_S10.get_foreground_power(S10_tSZ_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}S10_patches/tSZ_90GHz_2x2deg_ra=6_dec=6_S10spectra.npy", {"l": S10_tSZ_ells, "dl": S10_tSZ_dls})

# radio
S10_radio_patch = foregrounds_S10.generate_discrete_foreground(
                                    frequency = 90,
                                    catalog = pd.read_csv(f"{overall_path_example}S10_patches/radio_2x2deg_source_catalog_ra=6_dec=6.csv"))
S10_radio_dls, S10_radio_ells = spectra_S10.get_foreground_power(S10_radio_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}S10_patches/radio_90GHz_2x2deg_ra=6_dec=6_S10spectra.npy", {"l": S10_radio_ells, "dl": S10_radio_dls})

# CIB
S10_CIB_patch = foregrounds_S10.generate_discrete_foreground(
                                    frequency = 90,
                                    catalog = pd.read_csv(f"{overall_path_example}S10_patches/CIB_2x2deg_source_catalog_ra=6_dec=6.csv"),
                                    scaling_factor = 0.75)
S10_CIB_dls, S10_CIB_ells = spectra_S10.get_foreground_power(S10_CIB_patch, mbb_inv, binning_file, deconvolve_pw = True)
np.save(f"{overall_path_example}S10_patches/CIB_90GHz_2x2deg_ra=6_dec=6_S10spectra.npy", {"l": S10_CIB_ells, "dl": S10_CIB_dls})
    
# kappa
mbb_inv, binning_file, Bbl = spectra_S10_kappa.get_binning_files(path = f"{overall_path_example}binning_files/", type_Cl = True)
S10_kappa_patch = foregrounds_S10_kappa.generate_diffuse_foreground(
                    component = 'kappa',
                    frequency = None,
                    fullsky_deconvolved_path = f"S10_data/deconvolved_fullsky/kappa_fullsky_deconvolved")
S10_kappa_cls, S10_kappa_ells = spectra_S10_kappa.get_foreground_power(S10_kappa_patch, mbb_inv, binning_file, deconvolve_pw = False, type_Cl = True)
np.save(f"{overall_path_example}S10_patches/kappa_2x2deg_ra=6_dec=6_S10spectra.npy", {"l": S10_kappa_ells, "cl": S10_kappa_cls})
'''

We also need the lensed CMB theory, which is reliant on the power of our custom kappa map:

In [ ]:
'''
data = CMB_patch.make_lensed_theory(HD_kappa_spectrum = np.load(f"{overall_path_example}kappa_2x2deg_ra=6_dec=6_spectra.npy", allow_pickle=True)[()])

np.savetxt(f'{overall_path_example}camb_full_output_lensed.dat', data, fmt='%.6f', delimiter='\t')
'''

#### Fig. 2

#### Fig. 3

#### Fig. 4